# 01. Modele nieparametryczne

Notebook generuje wyniki nieparametrycznej części prezentacji: cenzurowanie, Kaplan-Meier, tablice aktuarialne, Nelson-Aalen, podgrupy oraz testy log-rank i Wilcoxona.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, NelsonAalenFitter, WeibullAFTFitter, LogNormalAFTFitter, LogLogisticAFTFitter, ExponentialFitter, CoxPHFitter
from lifelines.statistics import logrank_test, proportional_hazard_test

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "Data" / "heart_failure_clinical_records_dataset.csv"
WORKSPACE = BASE_DIR
TABLES = WORKSPACE / "Tables"
FIGS = WORKSPACE / "Wykresy"

df = pd.read_csv(DATA_PATH).drop_duplicates()
df["duration"] = df["time"].astype(float)
df["event"] = df["DEATH_EVENT"].astype(int)
df["ef_low"] = (df["ejection_fraction"] < 35).astype(int)
df["creatinine_high"] = (df["serum_creatinine"] > 1.5).astype(int)
df["sodium_low"] = (df["serum_sodium"] < 135).astype(int)
df["log_cpk"] = np.log1p(df["creatinine_phosphokinase"])
df.head()

In [ ]:
# Struktura zdarzeń i cenzurowania
df['event'].value_counts().rename({1:'zgon', 0:'cenzurowanie'})

In [ ]:
# Krzywa Kaplana-Meiera
kmf = KaplanMeierFitter().fit(df['duration'], df['event'])
ax = kmf.plot_survival_function(ci_show=True)
ax.set(xlabel='Czas (dni)', ylabel='S(t)', title='Kaplan-Meier')
plt.show()

In [ ]:
# Estymator Nelsona-Aalena
naf = NelsonAalenFitter().fit(df['duration'], df['event'])
ax = naf.plot_cumulative_hazard(ci_show=True)
ax.set(xlabel='Czas (dni)', ylabel='H(t)', title='Nelson-Aalen')
plt.show()

In [ ]:
# Wykresy KM dla podgrup
for var in ['ef_low','creatinine_high','sodium_low','high_blood_pressure','anaemia','diabetes','sex','smoking']:
    ax = plt.gca()
    for value, grp in df.groupby(var):
        KaplanMeierFitter(label=f'{var}={value}').fit(grp['duration'], grp['event']).plot_survival_function(ax=ax, ci_show=False)
    ax.set(title=var, xlabel='Czas (dni)', ylabel='S(t)')
    plt.show()

In [ ]:
# Testy log-rank i Wilcoxona
rows=[]
for var in ['ef_low','creatinine_high','sodium_low','high_blood_pressure','anaemia','diabetes','sex','smoking']:
    a,b=df[df[var]==0],df[df[var]==1]
    lr=logrank_test(a.duration,b.duration,event_observed_A=a.event,event_observed_B=b.event)
    wx=logrank_test(a.duration,b.duration,event_observed_A=a.event,event_observed_B=b.event,weightings='wilcoxon')
    rows.append([var,len(a),a.event.sum(),len(b),b.event.sum(),lr.test_statistic,lr.p_value,wx.test_statistic,wx.p_value])
pd.DataFrame(rows, columns=['zmienna','N0','zgony0','N1','zgony1','LR chi2','p LR','Wilcoxon chi2','p Wilcoxon'])

Hipoteza zerowa testów: funkcje przeżycia w porównywanych grupach są identyczne w całym zakresie czasu obserwacji.